In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
os.chdir("/Users/kanhapandey/Projects/SIH_PAIMANA")

In [ ]:
df = pd.read_csv("data/raw/monthly_completed_projects.csv")

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.drop("Sl.No", axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
def split_name_agency_code(text: str):
    """Split 'Project Name (Agency) Code' into (project_name, agency, project_code),
    correctly handling an Agency that itself contains parentheses."""
    text = text.strip()

    # Project code is the last whitespace-separated token
    remainder, _, code_part = text.rpartition(' ')
    remainder = remainder.strip()

    # remainder now ends with '...(Agency)' — walk back from the final ')'
    # counting parenthesis balance to find its matching '('
    close_idx = len(remainder) - 1
    balance = 0
    open_idx = None
    for i in range(close_idx, -1, -1):
        ch = remainder[i]
        if ch == ')':
            balance += 1
        elif ch == '(':
            balance -= 1
            if balance == 0:
                open_idx = i
                break

    agency = remainder[open_idx + 1:close_idx].strip()
    project_name = remainder[:open_idx].strip()
    return pd.Series([project_name, agency, code_part])


In [ ]:
df[['project_name', 'agency', 'project_code']] = (
    df['Project Name (Agency) (Project Code)'].apply(split_name_agency_code)
)

df.head()

In [ ]:
df.drop("Project Name (Agency) (Project Code)", axis =1, inplace=True)

In [ ]:
df.head()

In [ ]:
parts = df['Date of Approval (Start Date) MM/YYYY'].str.replace(')', '', regex=False).str.split('(', expand=True)
df['date_of_approval'] = parts[0].str.strip().replace('', np.nan)
df['start_date'] = parts[1].str.strip()
df.head()

In [ ]:
df.drop("Date of Approval (Start Date) MM/YYYY", axis =1, inplace=True)
df.head()

In [ ]:
parts = (
    df['Actual Date of Completion (Orignal/Target DoC) (Revised DoC) MM/YYYY']
    .str.replace(')', '', regex=False)
    .str.split('(', expand=True)
)
df['actual_doc'] = parts[0].str.strip().replace('', np.nan)
df['target_doc'] = parts[1].str.strip()
df['revised_doc'] = parts[2].str.strip()
df.drop("Actual Date of Completion (Orignal/Target DoC) (Revised DoC) MM/YYYY", axis =1, inplace=True)
df.head()

In [ ]:
parts = df['Orignal Cost (Revised Cost) in Rs. Crore'].str.replace(')', '', regex=False).str.split('(', expand=True)
df['original_cost_cr'] = parts[0].str.strip()
df['revised_cost_cr'] = parts[1].str.strip()
df.drop("Orignal Cost (Revised Cost) in Rs. Crore", axis =1, inplace=True)
df.head()

In [ ]:
date_cols = ['date_of_approval', 'start_date', 'actual_doc', 'target_doc', 'revised_doc']
missing_tokens = {'-', 'NA', 'na', 'N/A', ''}

for c in date_cols:
    df[c] = df[c].replace(missing_tokens, np.nan)
    df[c] = pd.to_datetime(df[c], format='%m/%Y', errors='coerce')

In [ ]:
cost_cols = ['original_cost_cr', 'revised_cost_cr']
for c in cost_cols:
    df[c] = df[c].replace(missing_tokens, np.nan).astype(float)

In [ ]:
df[date_cols + cost_cols].dtypes

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df["Physical Progress (in percentage)"] = float(100)

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
ordered_cols = [
    'project_code', 'project_name', 'agency', 'State',
    'date_of_approval', 'start_date',
    'actual_doc', 'target_doc', 'revised_doc',
    'original_cost_cr', 'revised_cost_cr',
    'Cumulative Expenditure in Rs. Crore', 'Physical Progress (in percentage)', 'Ministry', 'Sector',
]
df = df[ordered_cols]
df.head()

In [ ]:
df["status"] = "Completed"
df.head()

In [ ]:
df.columns = df.columns.str.lower()
print(df.columns)

In [ ]:
df.info()

In [ ]:
df.to_csv("data/cleaned/montly_completed_projects_clean.csv", index=False)